In [ ]:
# Cell 1: Mount Google Drive, locate baseline folder, set working dir, check GPU
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    # Search in shared drives if needed
    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))
    raise FileNotFoundError("Could not find final_project/baseline in your Drive. Please verify the folder path.")

BASE_DIR = find_baseline_dir()
print("BASE_DIR =", BASE_DIR)

os.chdir(BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Cell 2 (FIXED): Install compatible HF stack (no bash)
import sys, subprocess

# (Optional but recommended) clean conflicting installs
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
                       "transformers", "tokenizers", "huggingface-hub"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",            # closer to repo requirements (optional but recommended)
    "huggingface-hub==0.21.4",  # IMPORTANT: must be >= 0.21.x for transformers 4.38.1
    "tokenizers==0.15.2",
    "transformers==4.38.1",
    "datasets==2.18.0",
    "evaluate==0.4.1",
    "safetensors==0.4.2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import huggingface_hub, tokenizers, transformers, numpy
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("Installed OK")

huggingface_hub: 0.21.4
tokenizers: 0.15.2
transformers: 4.38.1
numpy: 1.26.4
Installed OK


In [ ]:
# Cell 3: Create folder structure inside baseline and copy JSONL files to expected paths
import os, shutil

# Create required dirs (skip IIRC as requested)
os.makedirs(os.path.join(BASE_DIR, "DATA", "KG"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "DATA", "2WikiMQA"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "DATA", "HotpotQA", "MDR"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "configs"), exist_ok=True)

# Your dataset files are here:
src_train = os.path.join(BASE_DIR, "train_with_neg_v0.json")
src_val   = os.path.join(BASE_DIR, "val_with_neg_v0.json")

# Where the original code expects them:
dst_train = os.path.join(BASE_DIR, "DATA", "HotpotQA", "MDR", "train_with_neg_v0.json")
dst_val   = os.path.join(BASE_DIR, "DATA", "HotpotQA", "MDR", "val_with_neg_v0.json")

assert os.path.isfile(src_train), f"Missing: {src_train}"
assert os.path.isfile(src_val), f"Missing: {src_val}"

# Copy if not already there (keeps original loader logic unchanged)
if not os.path.isfile(dst_train):
    shutil.copyfile(src_train, dst_train)
if not os.path.isfile(dst_val):
    shutil.copyfile(src_val, dst_val)

print("Dataset ready at:")
print(" -", dst_train)
print(" -", dst_val)

Dataset ready at:
 - /content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/train_with_neg_v0.json
 - /content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json


In [ ]:
# Cell 4: Write MDR.yml (optional; helps match repo layout)
import os, yaml

args_dict = {
    "root_dir": ".",
    "train_percent": 1.0,
    "model_name": "deepset/tinyroberta-squad2",
    "max_len": 200,
    "max_q_len": 64,
    "max_q_sp_len": 256,
    "weight_decay": 0.0,
    "lr": 3e-5,
    "adam_epsilon": 1e-8,
    "freeze_layers": 0,
    "train_bsz": 32,
    "num_workers": 0,     # Colab-safe (change to 2/4 if you want faster)
    "eval_bsz": 32,
    "epochs": 10,
    "warm_ratio": 0.2,
    "max_grad_norm": 2.0,
    "seed": 1028,
    "gpus": "cuda",       # IMPORTANT for A100
    "do_train": True,
    "checkpoint_root_dir": "checkpoint_tinyRo",
    "checkpoint_id": 1,
    "checkpoint_step": 250,
    "from_checkpoint": None,
}

cfg_path = os.path.join(BASE_DIR, "configs", "MDR.yml")
with open(cfg_path, "w") as f:
    yaml.dump(args_dict, f, sort_keys=False)

print("Wrote:", cfg_path)

Wrote: /content/drive/MyDrive/final_project/baseline/configs/MDR.yml


In [ ]:
# Cell 5: Tokenizer + utils
import random
import numpy as np
import torch
from transformers import AutoConfig, AutoTokenizer

def load_tokenizer(model_name="bert-base-uncased"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    config = AutoConfig.from_pretrained(model_name)
    return tokenizer, config

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def move_to_gpu(sample, device):
    if len(sample) == 0:
        return {}

    def _move(maybe_tensor):
        if torch.is_tensor(maybe_tensor):
            return maybe_tensor.to(device)
        elif isinstance(maybe_tensor, dict):
            return {k: _move(v) for k, v in maybe_tensor.items()}
        else:
            return maybe_tensor

    return _move(sample)

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Cell 6: MDR Retriever model
import torch.nn as nn
from transformers import AutoModel
import torch

class Retriever(nn.Module):
    def __init__(self, config, args):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(args["model_name"])
        self.args = args
        self.freeze_encoder()
        self.project = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps),
        )

    def freeze_encoder(self):
        if self.args["freeze_layers"] > 0:
            model_config = self.encoder.config
            if self.args["freeze_layers"] >= model_config.num_hidden_layers:
                for param in self.encoder.parameters():
                    param.requires_grad = False
            else:
                for name, param in self.encoder.named_parameters():
                    layer_index = name.split(".")[2]
                    if name.startswith("encoder.layer") and int(layer_index) < self.args["freeze_layers"]:
                        param.requires_grad = False

    def encode_seq(self, input_ids, mask):
        cls_rep = self.encoder(input_ids, mask)[0][:, 0, :]
        vector = self.project(cls_rep)
        return vector

    def forward(self, batch):
        q_emb = self.encode_seq(batch["q_enc_btz"], batch["q_mask"])
        q_c1_emb = self.encode_seq(batch["q_c1_enc_btz"], batch["q_c1_mask"])

        c1_emb = self.encode_seq(batch["c1_enc_btz"], batch["c1_mask"])
        c2_emb = self.encode_seq(batch["c2_enc_btz"], batch["c2_mask"])

        n1_emb = self.encode_seq(batch["n1_enc_btz"], batch["n1_mask"])
        n2_emb = self.encode_seq(batch["n2_enc_btz"], batch["n2_mask"])

        return {
            "q_emb": q_emb, "q_c1_emb": q_c1_emb,
            "c1_emb": c1_emb, "c2_emb": c2_emb,
            "n1_emb": n1_emb, "n2_emb": n2_emb
        }

In [ ]:
# Cell 7: Dataset + collate
import json
from torch.utils.data import Dataset

def load_dataset(train_percent=1.0, seed=42):
    with open("./DATA/HotpotQA/MDR/train_with_neg_v0.json", "r") as file:
        train_data = [json.loads(line) for line in file if len(json.loads(line)["neg_paras"]) >= 2]

    with open("./DATA/HotpotQA/MDR/val_with_neg_v0.json", "r") as file:
        val_data = [json.loads(line) for line in file]

    print(f"Loaded {len(train_data)} train and {len(val_data)} val data")

    random.seed(seed)
    if train_percent < 1.0:
        train_data = random.sample(train_data, int(train_percent * len(train_data)))
        print(f"Sampled {len(train_data)} train data")
    else:
        print("Using all the training data")
    return train_data, val_data

class HotpotQANeg(Dataset):
    def __init__(self, data, tokenizer, args, train: bool):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_len = args["max_len"]
        self.max_q_len = args["max_q_len"]
        self.max_q_sp_len = args["max_q_sp_len"]
        self.data = data
        self.args = args
        self.train = train

    def encode_chunk_pair(self, chunk1, chunk2, max_len):
        return self.tokenizer(
            text=chunk1, text_pair=chunk2,
            max_length=max_len, return_tensors="pt",
            padding=True, truncation=True
        )

    def encode_chunk(self, chunk, max_len):
        return self.tokenizer(
            text=chunk,
            max_length=max_len, return_tensors="pt",
            padding=True, truncation=True
        )

    def __getitem__(self, index):
        d = self.data[index]
        question = d["question"]
        if question.endswith("?"):
            question = question[:-1]

        if d["type"] == "comparison":
            random.shuffle(d["pos_paras"])
            start_para, bridge_para = d["pos_paras"][0], d["pos_paras"][1]
        else:
            for para in d["pos_paras"]:
                if para["title"] != d["bridge"]:
                    start_para = para
                else:
                    bridge_para = para

        if self.train:
            random.shuffle(d["neg_paras"])

        c1_enc = self.encode_chunk_pair(start_para["title"].strip(), start_para["text"].strip(), self.max_len)
        c2_enc = self.encode_chunk_pair(bridge_para["title"].strip(), bridge_para["text"].strip(), self.max_len)

        n1_enc = self.encode_chunk_pair(d["neg_paras"][0]["title"].strip(), d["neg_paras"][0]["text"].strip(), self.max_len)
        n2_enc = self.encode_chunk_pair(d["neg_paras"][1]["title"].strip(), d["neg_paras"][1]["text"].strip(), self.max_len)

        q_enc = self.encode_chunk(question, max_len=self.max_q_len)
        q_c1_enc = self.encode_chunk_pair(question, start_para["text"].strip(), self.max_q_sp_len)

        return {
            "q_enc": q_enc, "q_c1_enc": q_c1_enc,
            "c1_enc": c1_enc, "c2_enc": c2_enc,
            "n1_enc": n1_enc, "n2_enc": n2_enc
        }

    def __len__(self):
        return len(self.data)

def collate_tokens(values, pad_idx, eos_idx=None, left_pad=False, move_eos_to_beginning=False):
    if len(values[0].size()) > 1:
        values = [v.view(-1) for v in values]
    size = max(v.size(0) for v in values)
    res = values[0].new(len(values), size).fill_(pad_idx)

    def copy_tensor(src, dst):
        assert dst.numel() == src.numel()
        if move_eos_to_beginning:
            assert src[-1] == eos_idx
            dst[0] = eos_idx
            dst[1:] = src[:-1]
        else:
            dst.copy_(src)

    for i, v in enumerate(values):
        copy_tensor(v, res[i][size - len(v):] if left_pad else res[i][:len(v)])
    return res

def Dataset_collate(samples):
    if len(samples) == 0:
        return {}

    batch = {
        "q_enc_btz": collate_tokens([s["q_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_mask": collate_tokens([s["q_enc"]["attention_mask"][0] for s in samples], 0),

        "q_c1_enc_btz": collate_tokens([s["q_c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_c1_mask": collate_tokens([s["q_c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c1_enc_btz": collate_tokens([s["c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c1_mask": collate_tokens([s["c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c2_enc_btz": collate_tokens([s["c2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c2_mask": collate_tokens([s["c2_enc"]["attention_mask"][0] for s in samples], 0),

        "n1_enc_btz": collate_tokens([s["n1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n1_mask": collate_tokens([s["n1_enc"]["attention_mask"][0] for s in samples], 0),

        "n2_enc_btz": collate_tokens([s["n2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n2_mask": collate_tokens([s["n2_enc"]["attention_mask"][0] for s in samples], 0),
    }
    return batch

In [ ]:
# Cell 8: Training utilities + run MDR training + save/copy model in baseline
import os, math, yaml
from datetime import datetime
from tqdm import tqdm
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from transformers import get_linear_schedule_with_warmup

def mp_loss(model, batch):
    embs = model(batch)
    loss_fct = CrossEntropyLoss(ignore_index=-1)

    c_embs = torch.cat([embs["c1_emb"], embs["c2_emb"]], dim=0)  # 2B x d
    n_embs = torch.cat([embs["n1_emb"].unsqueeze(1), embs["n2_emb"].unsqueeze(1)], dim=1)  # B x 2 x d

    scores_1 = torch.mm(embs["q_emb"], c_embs.t())  # B x 2B
    n_scores_1 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2
    scores_2 = torch.mm(embs["q_c1_emb"], c_embs.t())  # B x 2B
    n_scores_2 = torch.bmm(embs["q_c1_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2

    bsize = embs["q_emb"].size(0)
    scores_1_mask = torch.cat([torch.zeros(bsize, bsize), torch.eye(bsize)], dim=1).to(embs["q_emb"].device)
    scores_1 = scores_1.float().masked_fill(scores_1_mask.bool(), float("-inf")).type_as(scores_1)
    scores_1 = torch.cat([scores_1, n_scores_1], dim=1)
    scores_2 = torch.cat([scores_2, n_scores_2], dim=1)

    target_1 = torch.arange(embs["q_emb"].size(0)).to(embs["q_emb"].device)
    target_2 = torch.arange(embs["q_emb"].size(0)).to(embs["q_emb"].device) + embs["q_emb"].size(0)

    loss = loss_fct(scores_1, target_1) + loss_fct(scores_2, target_2)
    return loss

def mhop_eval(embs):
    c_embs = torch.cat([embs["c1_emb"], embs["c2_emb"]], dim=0)
    n_embs = torch.cat([embs["n1_emb"].unsqueeze(1), embs["n2_emb"].unsqueeze(1)], dim=1)

    scores_1 = torch.mm(embs["q_emb"], c_embs.t())
    n_scores_1 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)
    scores_2 = torch.mm(embs["q_c1_emb"], c_embs.t())
    n_scores_2 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)

    bsize = embs["q_emb"].size(0)
    scores_1_mask = torch.cat([torch.zeros(bsize, bsize), torch.eye(bsize)], dim=1).to(embs["q_emb"].device)
    scores_1 = scores_1.float().masked_fill(scores_1_mask.bool(), float("-inf")).type_as(scores_1)
    scores_1 = torch.cat([scores_1, n_scores_1], dim=1)
    scores_2 = torch.cat([scores_2, n_scores_2], dim=1)

    target_1 = torch.arange(embs["q_emb"].size(0)).to(embs["q_emb"].device)
    target_2 = torch.arange(embs["q_emb"].size(0)).to(embs["q_emb"].device) + embs["q_emb"].size(0)

    ranked_1_hop = scores_1.argsort(dim=1, descending=True)
    ranked_2_hop = scores_2.argsort(dim=1, descending=True)
    idx2ranked_1 = ranked_1_hop.argsort(dim=1)
    idx2ranked_2 = ranked_2_hop.argsort(dim=1)

    rrs_1, rrs_2 = [], []
    for t, idx2ranked in zip(target_1, idx2ranked_1):
        rrs_1.append(1 / (idx2ranked[t].item() + 1))
    for t, idx2ranked in zip(target_2, idx2ranked_2):
        rrs_2.append(1 / (idx2ranked[t].item() + 1))
    return {"rrs_1": rrs_1, "rrs_2": rrs_2}

@torch.no_grad()
def eval_mdr(model, dataloader, args):
    model.eval()
    rrs_1, rrs_2 = [], []
    for batch in tqdm(dataloader):
        batch = move_to_gpu(batch, device=args["gpus"])
        embs = model(batch)
        out = mhop_eval(embs)
        rrs_1 += out["rrs_1"]
        rrs_2 += out["rrs_2"]
    return np.mean(rrs_1), np.mean(rrs_2)

def train_mdr(train_data, val_data, model, tokenizer, collate, args, checkpoint_dir=None):
    train_dataset = HotpotQANeg(train_data, tokenizer, args, train=True)
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args["train_bsz"],
        pin_memory=True,
        collate_fn=collate,
        num_workers=args["num_workers"],
        shuffle=True,
    )

    val_dataset = HotpotQANeg(val_data, tokenizer, args, train=False)
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args["eval_bsz"],
        pin_memory=True,
        collate_fn=collate,
        num_workers=args["num_workers"],
        shuffle=False,
    )

    t_total = len(train_dataloader) * args["epochs"]
    warmup_steps = math.ceil(t_total * args["warm_ratio"])
    print("Start Training")

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    save_dir = f"./models/{args['checkpoint_root_dir']}/MDR/HotpotQA/{args['checkpoint_id']}/{timestamp}"
    os.makedirs(save_dir, exist_ok=True)

    with open(os.path.join(save_dir, "mdr_config.yml"), "w") as f:
        yaml.dump(args, f, sort_keys=False)

    epoch = 0
    best_mrr = 0
    batch_num = 0
    hist_losses = []

    no_decay = ["bias", "LayerNorm.weight"]

    if checkpoint_dir is not None:
        print("Resuming training from checkpoint...")
        checkpoint = torch.load(checkpoint_dir, map_location="cpu")
        model.load_state_dict(checkpoint["model_state_dict"])
        model = model.to(args["gpus"])

        optimizer_parameters = [
            {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
             "weight_decay": args["weight_decay"]},
            {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
             "weight_decay": 0.0},
        ]
        optimizer = Adam(optimizer_parameters, lr=float(args["lr"]), eps=float(args["adam_epsilon"]))
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total)
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        epoch = checkpoint["epoch"]
        batch_num = checkpoint["batch_num"]
        hist_losses = checkpoint["losses"]
        best_mrr = checkpoint["best_mrr"]

        print(f"Loaded checkpoint from epoch {epoch} batch {batch_num}")
    else:
        print("Starting training from scratch...")
        model = model.to(args["gpus"])
        optimizer_parameters = [
            {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
             "weight_decay": args["weight_decay"]},
            {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
             "weight_decay": 0.0},
        ]
        optimizer = Adam(optimizer_parameters, lr=float(args["lr"]), eps=float(args["adam_epsilon"]))
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total)

    if epoch >= args["epochs"]:
        print(f"Epoch {epoch} has already been trained")
        return save_dir

    model = model.to(args["gpus"])

    for epoch in range(epoch, args["epochs"]):
        model.train()
        progress = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch}")
        for idx, samples in progress:
            if batch_num != 0 and idx <= batch_num:
                continue

            batch_num = 0
            samples = move_to_gpu(samples, device=args["gpus"])
            loss = mp_loss(model, samples)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args["max_grad_norm"])
            optimizer.step()
            scheduler.step()
            model.zero_grad()

            if idx % 100 == 0:
                hist_losses.append(loss.item())
                print(f"Epoch: {epoch}, Batch: {idx}, Loss: {np.mean(hist_losses)}")

            if idx % args["checkpoint_step"] == 0 and idx != 0:
                print(f"Start Evaluation after {args['checkpoint_step']} steps...")
                mrr_1, mrr_2 = eval_mdr(model, val_dataloader, args)
                mrr_avg = (mrr_1 + mrr_2) / 2

                print(f"Epoch: {epoch}, Loss: {np.mean(hist_losses)}, MRR_1: {mrr_1}, MRR_2: {mrr_2}, Ave_MRR: {mrr_avg}, Best_MRR: {best_mrr}")

                model_paths = [os.path.join(save_dir, "model.pt")]
                if mrr_avg > best_mrr:
                    best_mrr = mrr_avg
                    print("Saving best model...")
                    model_paths.append(os.path.join(save_dir, "best_model.pt"))

                for path in model_paths:
                    torch.save({
                        "epoch": epoch,
                        "batch_num": idx,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "smoothed_loss": np.mean(hist_losses),
                        "losses": hist_losses,
                        "curr_mrr": mrr_avg,
                        "best_mrr": best_mrr
                    }, path)

    return save_dir

# ---- Run (matches MDR_main.py logic) ----
args = yaml.safe_load(open("./configs/MDR.yml", "r"))

seed_everything(args["seed"])
tokenizer, config = load_tokenizer(args["model_name"])

train_data, val_data = load_dataset(train_percent=args["train_percent"], seed=args["seed"])

model = Retriever(config, args)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"The model has {num_params} parameters.")

save_dir = train_mdr(
    train_data=train_data,
    val_data=val_data,
    model=model,
    tokenizer=tokenizer,
    collate=Dataset_collate,
    args=args,
    checkpoint_dir=args["from_checkpoint"]
)

print("Saved checkpoints in:", save_dir)

# Copy final/best model to baseline root for convenience
best_path = os.path.join(save_dir, "best_model.pt")
last_path = os.path.join(save_dir, "model.pt")
dst_best = os.path.join(BASE_DIR, "mdr_best_model.pt")
dst_last = os.path.join(BASE_DIR, "mdr_last_model.pt")

import shutil
if os.path.isfile(best_path):
    shutil.copyfile(best_path, dst_best)
    print("Copied:", dst_best)
if os.path.isfile(last_path):
    shutil.copyfile(last_path, dst_last)
    print("Copied:", dst_last)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch: 9, Loss: 0.7628826388382108, MRR_1: 0.9397497832574677, MRR_2: 0.9019181025468113, Ave_MRR: 0.9208339429021395, Best_MRR: 0.9217330565152784


Epoch 9:  99%|█████████▉| 2801/2827 [45:58<00:18,  1.40it/s]

Epoch: 9, Batch: 2800, Loss: 0.7602522431345587


Epoch 9: 100%|██████████| 2827/2827 [46:16<00:00,  1.02it/s]


Saved checkpoints in: ./models/checkpoint_tinyRo/MDR/HotpotQA/1/2026-02-21_14-20-33
Copied: /content/drive/MyDrive/final_project/baseline/mdr_best_model.pt
Copied: /content/drive/MyDrive/final_project/baseline/mdr_last_model.pt


In [ ]:
import os, glob

base = BASE_DIR
fixed_best = os.path.join(base, "mdr_best_model.pt")
fixed_last = os.path.join(base, "mdr_last_model.pt")

ts_dir = os.path.join(base, "models", "checkpoint_tinyRo", "MDR", "HotpotQA", "1")
candidates = glob.glob(os.path.join(ts_dir, "*", "best_model.pt")) + glob.glob(os.path.join(ts_dir, "*", "model.pt"))

print("fixed_best exists:", os.path.isfile(fixed_best))
print("fixed_last exists:", os.path.isfile(fixed_last))
print("timestamp checkpoints found:", len(candidates))
if candidates:
    print("example:", candidates[-1])

fixed_best exists: True
fixed_last exists: True
timestamp checkpoints found: 2
example: /content/drive/MyDrive/final_project/baseline/models/checkpoint_tinyRo/MDR/HotpotQA/1/2026-02-21_14-20-33/model.pt
